# bcNMF on blended MNIST

Set `SETTING` to `mnist_k2` for digits 0 versus 1 or `mnist_k16` for the ten-digit benchmark.

## 1. Setup

This cell finds the repository whether Jupyter starts from the repository root, `notebooks/`, or `demo/`.

In [1]:
from pathlib import Path
import sys

def repository_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Open this notebook from the bcNMF reproducibility repository.")

ROOT = repository_root()
sys.path.insert(0, str(ROOT / "src"))

import contextlib
import io
import anndata as ad
import numpy as np
import pandas as pd
import torch
sys.modules["tensorflow"] = None  # UMAP does not need TensorFlow for this analysis
import umap
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from bcnmf import contrastive_nmf_poisson, contrastive_nmf_sse

def dense(matrix):
    return np.ascontiguousarray(matrix.toarray() if hasattr(matrix, "toarray") else matrix, dtype=np.float64)

def umap_ari(H, labels, seed=42, n_neighbors=15, min_dist=0.1):
    embedding = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                          metric="euclidean", random_state=seed).fit_transform(H.T)
    clusters = KMeans(n_clusters=len(np.unique(labels)), random_state=seed, n_init=10).fit_predict(embedding)
    return adjusted_rand_score(labels, clusters), embedding

def paper_result(dataset):
    result = pd.read_csv(ROOT / "results/paper_ari_reference.csv")
    return result[(result["dataset"] == dataset) & (result["method"] == "bcNMF")][["ari_mean", "ari_sd"]]


/usr/local/pkgs/anaconda/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Data

The repository stores the labelled target and background pixel matrices used by the paper workflow.

In [2]:
SETTING = "mnist_k2"  # or "mnist_k16"
data = np.load(ROOT / f"data/mnist/{SETTING}.npz")
X = np.ascontiguousarray(data["X"].T, dtype=np.float64)
Y = np.ascontiguousarray(data["Y"].T, dtype=np.float64)
labels = np.asarray(data["labels"])
K = int(data["latent_k"])
N_CLUSTERS = int(data["n_clusters"])
dataset = "MNIST K=2" if SETTING == "mnist_k2" else "MNIST K=16"
print(f"target {X.shape}; background {Y.shape}; K={K}; clusters={N_CLUSTERS}")

target (784, 1477); background (784, 650); K=2; clusters=2


## 3. bcNMF

Squared-error bcNMF with alpha=100,000 and 500 iterations.

In [3]:
ALPHA, NITER, SEED = 100000.0, 500, 0
torch.manual_seed(SEED); np.random.seed(SEED)
with contextlib.redirect_stdout(io.StringIO()):
    W, H_X, H_Y, _ = contrastive_nmf_sse(X, Y, K, ALPHA, niter=NITER)
H = np.asarray(H_X)


## 4. ARI

The paper endpoint clusters the target coefficients.

In [4]:
clusters = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit_predict(H.T)
ari = adjusted_rand_score(labels, clusters)
print(f"{dataset} ARI = {ari:.4f}")
print("manuscript bcNMF result:")
display(paper_result(dataset))

MNIST K=2 ARI = 0.7974
manuscript bcNMF result:


,ari_mean,ari_sd
3,0.8628,0.022
